## weights-contributors.ipynb

Builds three sparse feature matrices from the album contributor parquet files
(produced by `mb_album_contributor_fact_duckdb.sql` and decoded by
`album_contributor.py`):

| Matrix | Description |
|---|---|
| `album_role_family_matrix.npz` | Weighted role-family profile per album |
| `album_instrument_matrix.npz` | Weighted instrument profile per album |
| `album_contributor_counts_matrix.npz` | Distinct contributor counts per role family |

All three are aligned to the master `album_ids.pkl` index.

### Source tables and confidence weights
The three parquet scopes are combined and each row is scaled by its
`confidence_weight` before aggregation:

| Scope | File | Confidence weight |
|---|---|---|
| Recording | `sql_feat_album_contributor_recording.parquet` | 1.0 |
| Work | `sql_feat_album_contributor_work.parquet` | 0.9 |
| Release | `sql_feat_album_contributor_release.parquet` | 0.6 |

In [4]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

# Parquet paths — decoded by album_contributor.py
RELEASE_PARQUET   = f'{DATA_DIR}/sql_feat_album_contributor_release.parquet'
RECORDING_PARQUET = f'{DATA_DIR}/sql_feat_album_contributor_recording.parquet'
WORK_PARQUET      = f'{DATA_DIR}/sql_feat_album_contributor_work.parquet'

In [5]:
# Load master album index
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums    = len(album_index)
print(f'Master album universe: {n_albums:,} albums')

Master album universe: 1,008,102 albums


In [7]:
# Load dictionary tables for decoding int IDs → strings
dict_role_family = pd.read_parquet(f'{DATA_DIR}/sql_feat_dict_role_family.parquet',
                                    columns=['role_family_id', 'family_value'])
dict_instrument  = pd.read_parquet(f'{DATA_DIR}/sql_feat_dict_instrument.parquet',
                                    columns=['instrument_id', 'instrument_value'])

# Load and concatenate all three scopes (int-encoded columns only)
COLS = ['release_group_id', 'artist_id', 'role_family_id',
        'instrument_id', 'confidence_weight']

frames = []
for path in [RECORDING_PARQUET, WORK_PARQUET, RELEASE_PARQUET]:
    frames.append(pd.read_parquet(path, columns=COLS))

df = pd.concat(frames, ignore_index=True)
df = df.rename(columns={'release_group_id': 'album_id'})

# Decode role_family_id → family_value string
df = df.merge(dict_role_family, on='role_family_id', how='left')
df = df.rename(columns={'family_value': 'role_family'})

# Decode instrument_id → instrument_value string (NULL rows stay NULL)
df = df.merge(dict_instrument, on='instrument_id', how='left')
df = df.rename(columns={'instrument_value': 'instrument_or_attr'})

# Keep only albums that exist in the master index
df = df[df['album_id'].isin(album_index)]

print(f'Total contributor rows loaded: {len(df):,}')
print(f'Unique albums with contributor data: {df["album_id"].nunique():,}')
print(f'Role families: {sorted(df["role_family"].dropna().unique())}')
print(f'Non-null instrument rows: {df["instrument_or_attr"].notna().sum():,}')

Total contributor rows loaded: 28,232,262
Unique albums with contributor data: 450,023
Role families: ['business_label', 'other', 'performance', 'production', 'technical', 'visual_packaging', 'writing']
Non-null instrument rows: 8,323,892


---
## Matrix 1: Role-family profile

For each album, sum the `confidence_weight` values per `role_family`, then
normalise so each album's profile sums to 1.0.  
This gives a distribution vector like:
```
{'performance': 0.6, 'production': 0.25, 'technical': 0.1, 'writing': 0.05}
```
Two albums with similar role-family distributions (e.g. both heavily
performance-driven with minimal production credits) will be pulled closer
together in cosine space.

In [8]:
# Aggregate: sum confidence_weight per (album_id, role_family)
rf_agg = (
    df.groupby(['album_id', 'role_family'], sort=False)['confidence_weight']
    .sum()
    .reset_index()
)

# Normalise per album so each row sums to 1.0
album_totals = rf_agg.groupby('album_id')['confidence_weight'].transform('sum')
rf_agg['weight_norm'] = (rf_agg['confidence_weight'] / album_totals).astype(np.float32)

# Encode role_family as dense sequential category
rf_cat  = pd.Categorical(rf_agg['role_family'])
n_roles = len(rf_cat.categories)

row_idx  = album_index.get_indexer(rf_agg['album_id'].values)
valid    = row_idx >= 0

X_role_family = csr_matrix(
    (rf_agg['weight_norm'].values[valid],
     (row_idx[valid], rf_cat.codes[valid])),
    shape=(n_albums, n_roles)
)

print(f'Role family categories : {list(rf_cat.categories)}')
print(f'X_role_family shape    : {X_role_family.shape}')
print(f'Non-zero entries       : {X_role_family.nnz:,}')

Role family categories : ['business_label', 'other', 'performance', 'production', 'technical', 'visual_packaging', 'writing']
X_role_family shape    : (1008102, 7)
Non-zero entries       : 1,155,920


---
## Matrix 2: Instrument profile

Same aggregation pattern as role-family, but over `instrument_or_attr`.
Null rows (credits with no instrument specified) are excluded.
Instruments appearing on fewer than 10 albums are dropped as noise
(same threshold used for tags in `weights-tags.ipynb`).

In [9]:
instr_df = df.dropna(subset=['instrument_or_attr']).copy()

# Drop rare instruments
instr_album_counts = instr_df.groupby('instrument_or_attr')['album_id'].nunique()
popular_instrs     = instr_album_counts[instr_album_counts >= 10].index
instr_df           = instr_df[instr_df['instrument_or_attr'].isin(popular_instrs)]

# Aggregate: sum confidence_weight per (album_id, instrument)
instr_agg = (
    instr_df.groupby(['album_id', 'instrument_or_attr'], sort=False)['confidence_weight']
    .sum()
    .reset_index()
)

# Normalise per album
instr_totals = instr_agg.groupby('album_id')['confidence_weight'].transform('sum')
instr_agg['weight_norm'] = (instr_agg['confidence_weight'] / instr_totals).astype(np.float32)

instr_cat    = pd.Categorical(instr_agg['instrument_or_attr'])
n_instrs     = len(instr_cat.categories)

row_idx  = album_index.get_indexer(instr_agg['album_id'].values)
valid    = row_idx >= 0

X_instrument = csr_matrix(
    (instr_agg['weight_norm'].values[valid],
     (row_idx[valid], instr_cat.codes[valid])),
    shape=(n_albums, n_instrs)
)

print(f'Instruments kept (>=10 albums): {n_instrs:,}')
print(f'X_instrument shape            : {X_instrument.shape}')
print(f'Non-zero entries              : {X_instrument.nnz:,}')

Instruments kept (>=10 albums): 591
X_instrument shape            : (1008102, 591)
Non-zero entries              : 1,459,235


---
## Matrix 3: Contributor counts per role family

Distinct number of artists per role family, min-max scaled to [0, 1].
Captures album scale — a large orchestral recording will have many
'performance' contributors; a solo electronic album will have very few.

In [10]:
from sklearn.preprocessing import MinMaxScaler

# Distinct artists per (album, role_family)
counts_agg = (
    df.groupby(['album_id', 'role_family'])['artist_id']
    .nunique()
    .reset_index()
    .rename(columns={'artist_id': 'n_contributors'})
)

# Pivot to wide: one column per role_family
counts_wide = (
    counts_agg
    .pivot(index='album_id', columns='role_family', values='n_contributors')
    .fillna(0)
    .astype(np.float32)
)

# Min-max scale each column independently
scaler       = MinMaxScaler()
counts_scaled = scaler.fit_transform(counts_wide.values)

# Align to master index
row_idx = album_index.get_indexer(counts_wide.index.values)
valid   = row_idx >= 0
n_cols  = counts_scaled.shape[1]

col_idx  = np.tile(np.arange(n_cols), valid.sum())
row_rep  = np.repeat(row_idx[valid], n_cols)
data     = counts_scaled[valid].ravel()

nz = data != 0.0
X_contributor_counts = csr_matrix(
    (data[nz], (row_rep[nz], col_idx[nz])),
    shape=(n_albums, n_cols)
)

print(f'Role families (columns): {list(counts_wide.columns)}')
print(f'X_contributor_counts shape: {X_contributor_counts.shape}')
print(f'Non-zero entries          : {X_contributor_counts.nnz:,}')

Role families (columns): ['business_label', 'other', 'performance', 'production', 'technical', 'visual_packaging', 'writing']
X_contributor_counts shape: (1008102, 7)
Non-zero entries          : 1,155,920


In [11]:
save_npz(f'{FEATURES_DIR}/album_role_family_matrix.npz',       X_role_family)
save_npz(f'{FEATURES_DIR}/album_instrument_matrix.npz',        X_instrument)
save_npz(f'{FEATURES_DIR}/album_contributor_counts_matrix.npz', X_contributor_counts)

print('Saved:')
print(f'  album_role_family_matrix.npz        {X_role_family.shape}')
print(f'  album_instrument_matrix.npz         {X_instrument.shape}')
print(f'  album_contributor_counts_matrix.npz {X_contributor_counts.shape}')

Saved:
  album_role_family_matrix.npz        (1008102, 7)
  album_instrument_matrix.npz         (1008102, 591)
  album_contributor_counts_matrix.npz (1008102, 7)
